# Deteccao de fraude em cartao de credito

**Objetivo:** apoiar a decisao de aprovar, revisar ou bloquear transacoes a partir de mineracao de dados.

**Fonte:** `mlg-ulb/creditcardfraud` (284.807 transacoes europeias, 492 fraudes = 0,17%).
Comando real usado para obter os dados:

```bash
kaggle datasets download mlg-ulb/creditcardfraud
```

Os dados brutos ficam em `data/` (gitignored, nunca versionados). O notebook le `data/creditcard.csv`.
Foram usados os dados reais completos (284k cabem em RAM folgada); se a RAM estourar,
use uma amostra estratificada de 50k (trecho comentado na celula de carga) e documente a troca.


In [1]:
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
while not (ROOT / "src" / "common.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.common import df_to_html_table, load_csv, save_fig
from src.mining import confusion_summary, run_kmeans, run_tree

SEED = 42

df = load_csv(str(ROOT / "data" / "creditcard.csv"))
# Se a RAM estourar, trocar a linha acima por amostra estratificada 50k:
# df = df.sample(frac=1, random_state=SEED)
# _, df = train_test_split(df, train_size=50_000, stratify=df["Class"], random_state=SEED)
print(f"linhas: {len(df)} | fraudes: {int(df['Class'].sum())} | taxa: {df['Class'].mean()*100:.3f}%")
print("colunas:", list(df.columns[:5]), "...", list(df.columns[-3:]))
print("nulos:", int(df.isnull().sum().sum()))
display(df.head(5))


linhas: 284807 | fraudes: 492 | taxa: 0.173%
colunas: ['Time', 'V1', 'V2', 'V3', 'V4'] ... ['V28', 'Amount', 'Class']
nulos: 0


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## 1. Entendimento dos dados

Dicionario: `Time` (segundos desde a 1a transacao), `V1..V28` (componentes PCA anonimizadas),
`Amount` (valor em euros) e `Class` (0 = legitima, 1 = fraude). Nao ha valores nulos.
O desafio central e o desbalanceamento extremo: so 0,17% das transacoes sao fraude,
logo acuracia isolada nao serve — a avaliacao usa matriz de confusao, precision/recall e custo.


In [2]:
legit = df[df["Class"] == 0]
fraud = df[df["Class"] == 1]
print(f"legitimas: {len(legit)} | fraudes: {len(fraud)}")
print("Amount mediana — legitima:", round(legit['Amount'].median(), 2),
      "| fraude:", round(fraud['Amount'].median(), 2))
print("Amount maximo — legitimo:", round(legit['Amount'].max(), 2),
      "| fraude:", round(fraud['Amount'].max(), 2))
display(df.groupby("Class")[["Time", "Amount"]].describe().round(2))

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["legitima (0)", "fraude (1)"], [len(legit), len(fraud)], color=["#1a5fb4", "#c01c28"])
ax.set_yscale("log")
for i, v in enumerate([len(legit), len(fraud)]):
    ax.text(i, v * 1.15, f"{v:,}".replace(",", "."), ha="center", fontsize=10)
ax.set_title("Desbalanceamento de classes (escala log)")
ax.set_ylabel("transacoes")
print("fig:", Path(save_fig(fig, "fraud_class_balance.png")).name)
plt.close(fig)


legitimas: 284315 | fraudes: 492
Amount mediana — legitima: 22.0 | fraude: 9.25
Amount maximo — legitimo: 25691.16 | fraude: 2125.87


Time                                                         \
          count      mean       std    min      25%      50%       75%   
Class                                                                    
0      284315.0  94838.20  47484.02    0.0  54230.0  84711.0  139333.0   
1         492.0  80746.81  47835.37  406.0  41241.5  75568.5  128483.0   

                   Amount                                                      
            max     count    mean     std  min   25%    50%     75%       max  
Class                                                                          
0      172792.0  284315.0   88.29  250.11  0.0  5.65  22.00   77.05  25691.16  
1      170348.0     492.0  122.21  256.68  0.0  1.00   9.25  105.89   2125.87

fig: fraud_class_balance.png


![Desbalanceamento de classes](assets/img/fraud_class_balance.png)

**Leitura:** 284.315 transacoes legitimas contra apenas 492 fraudes (0,17%).
Qualquer modelo precisa de `class_weight=balanced` (ou reamostragem) para nao ignorar a classe rara.


In [3]:
fig, ax = plt.subplots(figsize=(7, 4))
bins = np.logspace(np.log10(1), np.log10(df["Amount"].max()), 60)
ax.hist(legit["Amount"], bins=bins, alpha=0.6, label="legitima", color="#1a5fb4")
ax.hist(fraud["Amount"], bins=bins, alpha=0.7, label="fraude", color="#c01c28")
ax.set_xscale("log")
ax.set_title("Distribuicao do valor (Amount, eixo x log)")
ax.set_xlabel("Amount (euros, log)")
ax.set_ylabel("transacoes")
ax.legend()
print("fig:", Path(save_fig(fig, "fraud_amount_dist.png")).name)
plt.close(fig)


fig: fraud_amount_dist.png


![Distribuicao do Amount](assets/img/fraud_amount_dist.png)

**Leitura:** fraudes concentram-se em valores baixos/medios (mediana menor que a das legitimas),
mas ha cauda de valores altos — valor sozinho nao separa as classes.


In [4]:
tmp = df.copy()
tmp["hour"] = ((tmp["Time"] // 3600) % 24).astype(int)
g = tmp.groupby("hour")
vol = g.size()
rate = g["Class"].mean() * 100

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.bar(vol.index, vol.values, color="#1a5fb4", alpha=0.6, label="volume")
ax1.set_xlabel("hora do dia")
ax1.set_ylabel("transacoes", color="#1a5fb4")
ax2 = ax1.twinx()
ax2.plot(rate.index, rate.values, color="#c01c28", marker="o", label="taxa de fraude %")
ax2.set_ylabel("taxa de fraude (%)", color="#c01c28")
ax1.set_title("Volume e taxa de fraude por hora do dia")
print("fig:", Path(save_fig(fig, "fraud_time_dist.png")).name)
plt.close(fig)
print("hora com maior taxa de fraude:", int(rate.idxmax()), f"({rate.max():.3f}%)")


fig: fraud_time_dist.png
hora com maior taxa de fraude: 2 (1.713%)


![Fraude por hora](assets/img/fraud_time_dist.png)

**Leitura:** o volume cai de madrugada, mas a taxa de fraude sobe — padrao util para regras destep-up noturno combinadas ao escore do modelo.


In [5]:
vcols = [f"V{i}" for i in range(1, 29)]
diff = (fraud[vcols].mean() - legit[vcols].mean()).sort_values(key=np.abs)
fig, ax = plt.subplots(figsize=(7, 6))
colors = ["#c01c28" if abs(v) > 1.0 else "#1a5fb4" for v in diff.values]
ax.barh(diff.index, diff.values, color=colors)
ax.set_title("Diferenca de medias fraude − legitima por componente V")
ax.set_xlabel("diferenca de medias")
print("fig:", Path(save_fig(fig, "fraud_v_features.png")).name)
plt.close(fig)
print("maiores desvios:", list(diff.abs().sort_values(ascending=False).head(5).index))


fig: fraud_v_features.png
maiores desvios: ['V3', 'V14', 'V17', 'V12', 'V10']


![Componentes V por classe](assets/img/fraud_v_features.png)

**Leitura:** varias componentes V tem medias bem distintas entre fraude e legitima
(barras vermelhas, |desvio| > 1) — ha sinal exploravel para clustering e classificacao.


## 2. Mineracao: perfis com KMeans

`run_kmeans` (k=4) sobre subamostra estratificada de 10k transacoes padronizadas,
para descobrir perfis de comportamento sem usar o rotulo.


In [6]:
FEATURES = [c for c in df.columns if c != "Class"]
sub, _ = train_test_split(df, train_size=10_000,
                              stratify=df["Class"], random_state=SEED)
y_sub = sub["Class"].reset_index(drop=True)
Xs = StandardScaler().fit_transform(sub[FEATURES])
labels, km = run_kmeans(pd.DataFrame(Xs), k=4, seed=SEED)

prof = sub.copy()
prof["cluster"] = labels
prof["Class"] = y_sub.values
prof_tab = prof.groupby("cluster").agg(n=("Class", "size"),
                                       fraude_pct=("Class", lambda s: round(100 * s.mean(), 2)),
                                       amount_medio=("Amount", lambda s: round(s.mean(), 2)))
display(prof_tab)

pca = PCA(n_components=2, random_state=SEED).fit_transform(Xs)
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(pca[:, 0], pca[:, 1], c=labels, cmap="tab10", s=6, alpha=0.4)
mask = y_sub.values == 1
ax.scatter(pca[mask, 0], pca[mask, 1], c="red", s=25, marker="x", label="fraude real")
ax.set_title("KMeans k=4 sobre PCA-2D (x vermelho = fraude)")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend()
print("fig:", Path(save_fig(fig, "fraud_kmeans.png")).name)
plt.close(fig)


,n,fraude_pct,amount_medio
cluster,,,
0,4169,0.14,59.76
1,4930,0.14,66.46
2,701,0.43,48.85
3,200,0.50,1270.35


fig: fraud_kmeans.png


![KMeans PCA](assets/img/fraud_kmeans.png)

**Leitura:** o KMeans separa um bolsao denso de transacoes legitimas e espalha as fraudes
(x vermelho) por micro-clusters — clustering sozinho nao detecta fraude, mas gera perfis
(volume, ticket medio, concentracao de risco) uteis para segmentar revisao manual.


## 3. Mineracao: classificacao supervisionada

`run_tree` como baseline + regressao logistica com `class_weight=balanced` (gera escore
probabilistico para a analise de limiar). Avaliacao com `confusion_summary` (matriz de confusao).
Split 70/30 estratificado; `Time`/`Amount` padronizados.


In [7]:
X = df[FEATURES].copy()
X["Time"] = StandardScaler().fit_transform(df[["Time"]])
X["Amount"] = StandardScaler().fit_transform(df[["Amount"]])
X_train, X_test, y_train, y_test = train_test_split(
    X, df["Class"], test_size=0.30, stratify=df["Class"], random_state=SEED)
print("treino:", X_train.shape, "| teste:", X_test.shape,
      "| fraudes no teste:", int(y_test.sum()))

tree = run_tree(X_train, y_train)
logr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)
logr.fit(X_train, y_train)

res_tree = confusion_summary(y_test, tree.predict(X_test))
res_log = confusion_summary(y_test, logr.predict(X_test))
print("arvore:", res_tree)
print("logistica balanced:", res_log)
for nome, yt in [("arvore", tree.predict(X_test)), ("logistica", logr.predict(X_test))]:
    p, r, f, _ = precision_recall_fscore_support(y_test, yt, average="binary", zero_division=0)
    print(f"{nome}: precision={p:.3f} recall={r:.3f} f1={f:.3f}")

cm = np.array(res_log["confusion_matrix"])
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i, j]:,}".replace(",", "."), ha="center", fontsize=12)
ax.set_xticks([0, 1], ["pred legitima", "pred fraude"])
ax.set_yticks([0, 1], ["real legitima", "real fraude"])
ax.set_title(f"Matriz de confusao — logistica (acuracia {res_log['accuracy']:.4f})")
print("fig:", Path(save_fig(fig, "fraud_confusion.png")).name)
plt.close(fig)


treino: (199364, 30) | teste: (85443, 30) | fraudes no teste: 148


arvore: {'accuracy': 0.9991807403766253, 'confusion_matrix': [[85264, 31], [39, 109]]}
logistica balanced: {'accuracy': 0.9786056201210164, 'confusion_matrix': [[83485, 1810], [18, 130]]}
arvore: precision=0.779 recall=0.736 f1=0.757
logistica: precision=0.067 recall=0.878 f1=0.125
fig: fraud_confusion.png


![Matriz de confusao](assets/img/fraud_confusion.png)

**Leitura:** a matriz de confusao mostra o dilema — com o limiar padrao 0,50 a logistica
balanceada captura a maioria das fraudes ao custo de falsos positivos. O ponto otimo
depende do custo de cada erro: e a analise de limiar (threshold) abaixo.


## 4. Analise de limiar (threshold): precision/recall vs custo

Varrer o limiar de decisao do escore e medir precision, recall e custo esperado
(FP = revisao manual, custo 10; FN = fraude perdida, custo 500). O limiar recomendado
minimiza o custo total no conjunto de teste.


In [8]:
proba = logr.predict_proba(X_test)[:, 1]
C_FP, C_FN = 10, 500
ths = np.concatenate([np.arange(0.05, 0.96, 0.05), [0.96, 0.97, 0.98, 0.99]])
rows = []
for t in ths:
    pred = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    p, r, f, _ = precision_recall_fscore_support(y_test, pred, average="binary", zero_division=0)
    rows.append({"threshold": round(float(t), 2), "precision": round(float(p), 3),
                 "recall": round(float(r), 3), "f1": round(float(f), 3),
                 "FP": int(fp), "FN": int(fn),
                 "custo": int(fp * C_FP + fn * C_FN)})
sweep = pd.DataFrame(rows)
display(sweep)
print(df_to_html_table(sweep, max_rows=25)[:80])

REC = sweep.loc[sweep["custo"].idxmin()]
T_REC = float(REC["threshold"])
print(f"Limiar recomendado: {T_REC:.2f} | precision={REC['precision']:.3f} "
      f"recall={REC['recall']:.3f} FP={int(REC['FP'])} FN={int(REC['FN'])} custo={int(REC['custo'])}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sweep["threshold"], sweep["precision"], marker="o", label="precision")
ax.plot(sweep["threshold"], sweep["recall"], marker="s", label="recall")
ax.plot(sweep["threshold"], sweep["f1"], marker="^", label="f1")
ax.axvline(T_REC, color="red", linestyle="--", label=f"limiar recomendado {T_REC:.2f}")
ax.set_title("Precision/recall/F1 por limiar (threshold)")
ax.set_xlabel("threshold")
ax.legend()
print("fig:", Path(save_fig(fig, "fraud_precision_recall.png")).name)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(sweep["threshold"].astype(str), sweep["custo"], color="#1a5fb4")
ax.bar(str(f"{T_REC:.2f}"), int(REC["custo"]), color="#c01c28")
ax.set_title(f"Custo esperado por limiar (FP=10, FN=500) — minimo em {T_REC:.2f}")
ax.set_xlabel("threshold")
ax.set_ylabel("custo total no teste")
plt.xticks(rotation=45)
print("fig:", Path(save_fig(fig, "fraud_cost.png")).name)
plt.close(fig)


,threshold,precision,recall,f1,FP,FN,custo
0,0.05,0.006,0.953,0.011,24243,7,245930
1,0.10,0.010,0.939,0.020,13855,9,143050
2,0.15,0.015,0.932,0.029,9359,10,98590
3,0.20,0.019,0.912,0.038,6796,13,74460
4,0.25,0.025,0.905,0.049,5195,14,58950
5,0.30,0.031,0.899,0.061,4091,15,48410
6,0.35,0.038,0.885,0.074,3278,17,41280
7,0.40,0.046,0.878,0.088,2667,18,35670
8,0.45,0.056,0.878,0.105,2196,18,30960
9,0.50,0.067,0.878,0.125,1810,18,27100


<table class="dataframe data-table">
  <thead>
    <tr style="text-align: right;
Limiar recomendado: 0.98 | precision=0.562 recall=0.831 FP=96 FN=25 custo=13460
fig: fraud_precision_recall.png


fig: fraud_cost.png


![Precision recall por limiar](assets/img/fraud_precision_recall.png)

![Custo por limiar](assets/img/fraud_cost.png)

**Leitura:** precision sobe e recall cai com o threshold; o custo total e minimo no limiar
recomendado acima (barra vermelha), que privilegia recall porque perder uma fraude (FN=500)
custa 50x mais que uma revisao manual (FP=10).


## 5. Decisao: limiar recomendado e regra de atuacao

**Regra de decisao:** SE escore > limiar recomendado ENTAO step-up
(autenticacao reforcada ou bloqueio preventivo + revisao), SENAO aprovar com monitoramento.
O valor exato do limiar esta impresso na secao 4 (variavel `T_REC`).

**Recomendacoes:**
1. Recomenda-se operar no limiar de custo minimo (alto recall): cada fraude evitada paga
   dezenas de revisoes manuais, dado o custo FN >> FP.
2. Recomenda-se step-up adaptativo de madrugada/horarios de alta taxa de fraude (secao 1):
   mesmo escore medio a noite deve exigir autenticacao reforcada.
3. Recomenda-se fila de revisao priorizada por escore nos micro-clusters de risco do KMeans,
   em vez de revisao aleatoria.
4. Recomenda-se recalibrar o limiar mensalmente: a distribuicao de fraude muda e o ponto
   de custo minimo se desloca; monitorar precision/recall em producao.


In [9]:
OUT = ROOT / "docs" / "assets" / "data"
OUT.mkdir(parents=True, exist_ok=True)
f = df[df["Class"] == 1].sample(100, random_state=SEED)
l = df[df["Class"] == 0].sample(100, random_state=SEED)
sample = pd.concat([f, l]).sample(frac=1, random_state=SEED).round(4)
sample.to_csv(OUT / "sample_fraud.csv", index=False)
size = (OUT / "sample_fraud.csv").stat().st_size
print(f"sample_fraud.csv: {len(sample)} linhas (100 fraude + 100 legitimas, oversampling "
      f"documentado — no real sao 0,17%), {size/1024:.1f} KB (limite 200 KB)")


sample_fraud.csv: 200 linhas (100 fraude + 100 legitimas, oversampling documentado — no real sao 0,17%), 44.0 KB (limite 200 KB)
